# Week 5.2 — Ridge Regression via Matrix-Free Conjugate Gradient
Solves a $\ell_2$-regularized (ridge) least-squares problem by applying CG directly to its normal equations, without ever forming the $d\times d$ Hessian $X^TX+\lambda I$ explicitly — only matrix-vector products with $X$ and $X^T$ are needed, which is what makes CG attractive for large, high-dimensional regression.

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import cg, LinearOperator

## Generate synthetic data
Creates a synthetic regression problem with more samples than features ($n=1500 \gg d=400$), a sparse true weight vector `w_true` (only 5% of entries nonzero), and noisy observations $y = Xw_{\text{true}} + \text{noise}$.

In [2]:
np.random.seed(0)
n = 1500    # samples
d = 400     # features
X = np.random.randn(n, d)

# Use scipy.sparse.random for sprandn equivalent
w_true_sparse = sp.random(d, 1, density=0.05, data_rvs=np.random.randn)
w_true = w_true_sparse.toarray().flatten()

y = X @ w_true + 0.1 * np.random.randn(n)

lambda_reg = 1e-1

## Matrix-free Hessian-vector product
The ridge regression normal equations are $(X^TX+\lambda I)w = X^Ty$. Rather than forming $X^TX$ (a dense $d\times d$ matrix), `hessian_vec_prod` computes $(X^TX+\lambda I)v$ as two matrix-vector products, $X^T(Xv)+\lambda v$, wrapped in a `LinearOperator` so `cg` never sees the matrix explicitly.

In [3]:
def hessian_vec_prod(v):
    return X.T @ (X @ v) + lambda_reg * v

A_op = LinearOperator((d, d), matvec=hessian_vec_prod, rmatvec=hessian_vec_prod)
b = X.T @ y

tol = 1e-8
maxit = 500

## Solve with CG
Since $X^TX+\lambda I$ is SPD for any $\lambda>0$, CG is directly applicable and converges to the ridge-regression solution; a callback counts the iterations actually taken.

In [4]:
# To get the iteration count, we can use a callback.
iter_count = [0]
def callback(xk):
    iter_count[0] += 1

x_cg, info = cg(A_op, b, rtol=tol, maxiter=maxit, callback=callback)

final_res_norm = np.linalg.norm(b - A_op @ x_cg)
b_norm = np.linalg.norm(b)
relres = final_res_norm / b_norm if b_norm > 0 else 0.0

print(f'CG info={info}, relres={relres:.2e}, iters={iter_count[0]}')

CG info=0, relres=9.10e-09, iters=26


## Compare to direct solve
Sanity-checks the matrix-free CG solution against explicitly forming $X^TX+\lambda I$ and solving with a direct dense solver — feasible here only because $d=400$ is small, unlike in a realistic large-scale setting.

In [5]:
x_dir = np.linalg.solve(X.T @ X + lambda_reg * np.eye(d), b)

print(f'||x_cg - x_dir||/||x_dir|| = {np.linalg.norm(x_cg - x_dir)/np.linalg.norm(x_dir):.2e}')

||x_cg - x_dir||/||x_dir|| = 2.03e-08
